<a href="https://colab.research.google.com/github/asrazia68-svg/Movie_review_sentiment_analysis.ipynb/blob/main/Movie__Review_Sentiment_Analysis_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import numpy as np
import pandas as pd
df = pd.read_csv('/content/IMDB Dataset.csvs')
df.head()
# 2. Map sentiments to numerical values as per assignment
# Positive -> 1, Negative -> -1
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': -1, 'neutral' : 0})

print(df.head())
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download necessary NLTK data
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # 1. Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # 2. Remove non-alphabetic characters and lowercase
    text = re.sub(r'[^a-zA-Z]', ' ', text).lower()
    # 3. Tokenize and remove stopwords + Lemmatization
    words = text.split()
    cleaned_words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]

    return " ".join(cleaned_words)

# Apply cleaning to the review column
df['cleaned_review'] = df['review'].apply(clean_text)

print(df[['review', 'cleaned_review']].head())
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# 1. Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=5000) # Hum top 5000 words use karein gay

# 2. Transform the cleaned text into numbers
X = tfidf.fit_transform(df['cleaned_review']).toarray()
y = df['sentiment']

# 3. Split data into Training and Testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print("Data transformation complete!")
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

# 1. Initialize the SVM Model with regularization
# LinearSVC fast hota hai large datasets ke liye
# 'C' inverse regularization strength hai. Kam 'C' ka matlab zyada regularization.
model = LinearSVC(random_state=42, C=0.01)

# 2. Train the model
model.fit(X_train, y_train)

# 3. Predict on the test set
y_pred = model.predict(X_test)

# 4. Results
print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))
def get_recommendation(review_text):
    # 1. Clean the input text
    clean_input = clean_text(review_text)

    # 2. Convert to numbers
    vectorized_input = tfidf.transform([clean_input])

    # 3. Predict Score (Predict ki jagah decision_function use karein)
    score = model.decision_function(vectorized_input)[0]

    # 4. Recommendation Logic with Threshold
    # Humne yahan -0.1 aur 0.1 ka ek gap rakha hai neutral ke liye
    if score > 0.1:
        prediction = 1
        recommendation = "Recommend movie"
    elif score < -0.1:
        prediction = -1
        recommendation = "Do not recommend"
    else:
        prediction = 0
        recommendation = "Maybe watch (Neutral)"

    return f"Score: {score:.2f}, Sentiment: {prediction}, Action: {recommendation}"

# --- Testing with Neutral Words ---
print(get_recommendation("good"))
print("Enter your review")
Recommendation = input("Enter your review:")

print(df['sentiment'].value_counts())
from sklearn.metrics import accuracy_score

# 1. Training set par prediction karein
y_train_pred = model.predict(X_train)
train_accuracy = accuracy_score(y_train, y_train_pred)

# 2. Testing set par prediction karein (Jo aap pehle kar chuke hain)
y_test_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_accuracy * 100:.2f}%")
print(f"Testing Accuracy:  {test_accuracy * 100:.2f}%")

# Overfitting check karne ke liye:
if train_accuracy > test_accuracy + 0.1:
    print("\nWarning: Model Overfit ho raha hai! (Training accuracy testing se kaafi zyada hai)")
else:
    print("\nModel achi tarah generalize ho raha hai.")

                                              review  sentiment
0  One of the other reviewers has mentioned that ...          1
1  A wonderful little production. <br /><br />The...          1
2  I thought this was a wonderful way to spend ti...          1
3  Basically there's a family where a little boy ...         -1
4  Petter Mattei's "Love in the Time of Money" is...          1


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                      cleaned_review  
0  one reviewer mentioned watching oz episode hoo...  
1  wonderful little production filming technique ...  
2  thought wonderful way spend time hot summer we...  
3  basically family little boy jake think zombie ...  
4  petter mattei love time money visually stunnin...  
X_train shape: (40000, 5000)
Data transformation complete!
Accuracy: 0.8734

Classification Report:

              precision    recall  f1-score   support

          -1       0.89      0.85      0.87      4961
           1       0.86      0.90      0.88      5039

    accuracy                           0.87     10000
   macro avg  